In [0]:
%pip install torch-geometric

In [0]:
# 1. Install the core engine
%pip install torch torchvision torchaudio

# 2. Install the graph libraries
%pip install torch-geometric

dbutils.library.restartPython()

In [0]:
'''
Basic Graph StructureNodes ($V$): Customers or Users.Edges ($E$): Interactions (Purchases, Referrals, Follows).Edge Index: A tensor of shape [2, num_edges] defining connectivity.
'''
import torch
import pandas as pd

# 1. SETUP DATA (The Customer Network)
# Format: [Source (Influencer), Target (Influenced)]
edges = torch.tensor([
    [0, 1, 2, 0, 4, 5], 
    [1, 2, 0, 3, 0, 0]
], dtype=torch.long)

def compute_customer_influence(edge_index, num_nodes=None, alpha=0.85, iterations=50):
    if num_nodes is None:
        num_nodes = edge_index.max().item() + 1
    
    # Initialize all customers with equal influence
    v = torch.ones((num_nodes, 1)) / num_nodes
    
    # Create the Adjacency Matrix
    # We use a sparse representation to save memory in Databricks
    adj = torch.zeros((num_nodes, num_nodes))
    for i in range(edge_index.shape[1]):
        src, tgt = edge_index[0, i], edge_index[1, i]
        adj[tgt, src] = 1.0  # Influence flows from src to tgt

    # Normalize columns (so each customer distributes their total influence)
    col_sum = adj.sum(dim=0)
    col_sum[col_sum == 0] = 1.0 # Avoid division by zero
    adj = adj / col_sum

    # Power Iteration: The heart of PageRank
    for _ in range(iterations):
        v = (1 - alpha) / num_nodes + alpha * torch.mm(adj, v)
        
    return v

# 2. EXECUTE
num_nodes = edges.max().item() + 1
influence_scores = compute_customer_influence(edges, num_nodes)

# 3. RESULTS (Format for Databricks Display)
results = pd.DataFrame({
    "customer_id": range(num_nodes),
    "influence_score": influence_scores.flatten().tolist()
}).sort_values(by="influence_score", ascending=False)

# Use the built-in Databricks display tool
display(spark.createDataFrame(results))

In [0]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
import pandas as pd

# 1. Define the Model Architecture
class InfluenceGNN(torch.nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.conv1 = GCNConv(num_features, 16)
        self.conv2 = GCNConv(16, 1) # Output a single influence score per node

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

# 2. Setup Data (Example: 6 customers)
# Customer connections (Who influences whom)
edge_index = torch.tensor([
    [0, 1, 2, 0, 4, 5], 
    [1, 2, 0, 3, 0, 0]
], dtype=torch.long)

num_nodes = edge_index.max().item() + 1

# Create features: In this case, we use an identity matrix (Each node is unique)
# This allows the GCN to act like a learnable version of PageRank
x = torch.eye(num_nodes) 

# Create the PyG Data object
data = Data(x=x, edge_index=edge_index)

# 3. Run Inference
model = InfluenceGNN(num_features=num_nodes)
model.eval() # Set to evaluation mode

with torch.no_grad():
    # Obtain the raw influence scores
    raw_scores = model(data)
    
    # Apply Sigmoid if you want scores between 0 and 1
    influence_scores = torch.sigmoid(raw_scores).squeeze()

# 4. Extract Top Influencers
results_pdf = pd.DataFrame({
    "customer_id": range(num_nodes),
    "influence_score": influence_scores.tolist()
}).sort_values(by="influence_score", ascending=False)

# Display in Databricks
display(spark.createDataFrame(results_pdf))

In [0]:
import networkx as nx
import matplotlib.pyplot as plt

# 1. Convert PyG edges to a NetworkX graph
edge_list = edge_index.t().tolist()
G = nx.DiGraph() # Directed Graph
G.add_edges_from(edge_list)

# 2. Prepare the Visualization
plt.figure(figsize=(10, 7))

# We use the influence_scores we calculated in the previous step
# Scale the scores so the nodes are visible (e.g., multiply by 2000)
node_sizes = [influence_scores[node].item() * 2000 for node in G.nodes()]

# 3. Draw the Graph
pos = nx.spring_layout(G, seed=42) # Layout for the nodes
nx.draw(
    G, pos, 
    with_labels=True, 
    node_size=node_sizes, 
    node_color="skyblue", 
    arrowsize=20, 
    edge_color="gray",
    font_weight="bold"
)

plt.title("Customer Influence Network (Node Size = Influence)")
plt.show() # This will render the graph directly in your Databricks notebook

In [0]:
%pip install pyvis

In [0]:
import torch
from pyvis.network import Network

# 1. DEFINE DATA (Run this even if you did before)
# Format: [Source IDs], [Target IDs]
edge_index = torch.tensor([
    [0, 1, 2, 0, 4, 5, 6, 7, 1], 
    [1, 2, 0, 3, 0, 0, 1, 1, 3]
], dtype=torch.long)

# 2. CALCULATE VARIABLES
num_nodes = edge_index.max().item() + 1

# If influence_scores aren't in memory, we create dummy ones 
# based on node degree for visualization purposes
if 'influence_scores' not in locals():
    # Simple count of how many people a node influences
    influence_scores = torch.bincount(edge_index[0], minlength=num_nodes).float()
    # Normalize
    influence_scores = influence_scores / (influence_scores.max() + 1e-6)

# 3. BUILD INTERACTIVE GRAPH
net = Network(notebook=True, height="600px", width="100%", bgcolor="#222222", font_color="white", directed=True, cdn_resources='remote')

for i in range(num_nodes):
    score = influence_scores[i].item()
    node_size = 15 + (score * 50) # Base size 15 + bonus for influence
    net.add_node(i, label=f"Customer {i}", size=node_size, title=f"Influence Score: {score:.2f}")

edges = edge_index.t().tolist()
net.add_edges(edges)

# Enable the physics engine so you can "watch" it move
net.toggle_physics(True)
net.show("influence_graph.html")

In [0]:
import IPython

# This reads the HTML file created by Pyvis and renders it in the output
with open("influence_graph.html", 'r') as f:
    html_content = f.read()

IPython.display.HTML(html_content)